In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier #use to handle tabular spatial weather data
from sklearn.metrics import classification_report, confusion_matrix #used to calculate overall accuracy and show the breakdown of the correct vs incorrect predictions
from sklearn.model_selection import LeaveOneGroupOut #used to split the data into training and testing sets based on a grouping variable
import joblib #used to save and load the trained model

In [2]:
# setup paths
ROOT=Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

label_path = ROOT / "data" / "labels" / "waterlogging_reports.csv"
print(f"Loading labels from: {label_path}")

lab = pd.read_csv(label_path)

Loading labels from: c:\Users\Ausaaf Siddiqui\AquaRoute-AI\data\labels\waterlogging_reports.csv


In [ ]:
# preprocess and feature engineering
lab["target"] = lab["severity"].apply(lambda x: 0 if x == "none" else 1)

severity_rank = {"none": 0, "moderate": 1, "high": 2}
lab["severity_rank"] = lab["severity"].map(severity_rank)
lab = lab.sort_values(by="severity_rank", ascending=False)
df = lab.drop_duplicates(subset=["date", "lat", "lon"], keep="first").copy()

# feature and target
X = df[["lat", "lon"]]
y = df["target"]
groups = df["date"] #leave one group out by event date

print(f"Dataset shape for training: {X.shape}")
print(f"Class distribution:\n{y.value_counts()}")


Dataset shape for training: (88, 2)
Class distribution:
target
0    63
1    25
Name: count, dtype: int64


In [7]:
#  Baseline Model Training using Leave-One-Event-Out Cross-Validation
logo = LeaveOneGroupOut()

print("\n Training Baseline Random Forest Model ")
for train_idx, test_idx in logo.split(X, y, groups=groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    test_date = groups.iloc[test_idx].iloc[0]

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    print(f"\nEvaluated on Held-Out Event Date: {test_date}")
    print(confusion_matrix(y_test, preds, labels=[0, 1]))

    best_model_path = ROOT / "models" / f"baseline_model_{test_date}.joblib"
    break  # Remove this break to evaluate on all events, currently only evaluating on the first event for demonstration

print("\n Baseline model test executed successfully!")


 Training Baseline Random Forest Model 

Evaluated on Held-Out Event Date: 2022-06-30
[[9 0]
 [0 0]]

 Baseline model test executed successfully!


In [12]:
# save the model artifacr for the api
models_dir = ROOT / "ml" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "waterlogging_rf_baseline.joblib"
    
    # Train final model on all data for deployment artifact
final_model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
final_model.fit(X, y)
    
joblib.dump(final_model, model_path)
print(f"\n Saved baseline model artifact to: {model_path}")
print("\n Baseline model training and persistence completed successfully!")




 Saved baseline model artifact to: c:\Users\Ausaaf Siddiqui\AquaRoute-AI\ml\models\waterlogging_rf_baseline.joblib

 Baseline model training and persistence completed successfully!
